# Chroma key extraction lab

This notebook is meant to debug why chroma-key removal fails on generated game-art layers.

It focuses on the typical failure modes:
- the chroma background is not perfectly uniform;
- anti-aliased borders mix object color with the chroma color;
- shadows or semi-transparent pixels near edges are wrongly classified;
- a fixed RGB threshold is too brittle;
- using a color present in the scene, such as green in vegetation, destroys useful pixels.

The recommended key color for the Bariloche bus-stop layers is **pure magenta**: `#FF00FF`.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Change this to the layer you want to inspect.
# IMAGE_PATH = Path("data/bariloche_bus_stop/01.png")
IMAGE_PATH = Path("data/julia/julia.png")

# Optional manual chroma-key override (RGB). Leave None to auto-detect from
# the image borders in the cell below. Useful when the borders contain
# content (a layer that bleeds to the edge) or are unusually noisy.
KEY_RGB_OVERRIDE = None  # np.array([255, 0, 255], dtype=np.uint8)

assert IMAGE_PATH.exists(), f"File not found: {IMAGE_PATH.resolve()}"


## Load image

In [ ]:
def imread_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

img = imread_rgb(IMAGE_PATH)
h, w = img.shape[:2]
print(img.shape)

plt.figure(figsize=(14, 7))
plt.imshow(img)
plt.axis("off");

In [ ]:
img[0:5,0:5].mean(axis=1)

## Estimate chroma-key color from borders

The four corner patches of a chroma-keyed render are almost always pure
background, so a per-channel median over them is a robust estimate. Override
above when the borders are atypical (a layer that bleeds to the edge,
content that touches a corner, very compressed source).


In [ ]:
def estimate_key_color(rgb: np.ndarray, patch: int = 20) -> np.ndarray:
    """Median chroma color sampled from the four corner patches."""
    h, w, _ = rgb.shape[:3]
    patch = max(1, min(patch, h // 2, w // 2))
    patches = [
        rgb[:patch, :patch],
        rgb[:patch, w - patch :],
        rgb[h - patch :, :patch],
        rgb[h - patch :, w - patch :],
    ]
    samples = np.concatenate([p.reshape(-1, 3) for p in patches], axis=0)
    return np.median(samples, axis=0).astype(np.uint8)


KEY_RGB = (
    KEY_RGB_OVERRIDE if KEY_RGB_OVERRIDE is not None else estimate_key_color(img)
)
print("Chroma key (RGB):", tuple(int(x) for x in KEY_RGB))

# Swatch + corner-patch preview, to sanity-check the inference.
PATCH = 20
h, w = img.shape[:2]
corners = np.concatenate(
    [
        np.concatenate([img[:PATCH, :PATCH], img[:PATCH, w - PATCH :]], axis=1),
        np.concatenate(
            [img[h - PATCH :, :PATCH], img[h - PATCH :, w - PATCH :]], axis=1
        ),
    ],
    axis=0,
)
swatch = np.full((PATCH * 2, PATCH * 2, 3), KEY_RGB[None, None, :], dtype=np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(corners)
axes[0].set_title("Stitched corner patches")
axes[0].axis("off")
axes[1].imshow(swatch)
axes[1].set_title(f"Inferred chroma RGB={tuple(int(x) for x in KEY_RGB)}")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## First diagnostic: how uniform is the chroma background?

This checks how far every pixel is from the expected key color in RGB space.  
If the background is truly flat `#FF00FF`, most background pixels should have distance near zero.

In [ ]:
key = KEY_RGB.astype(np.float32)
img_f = img.astype(np.float32)

rgb_dist = np.linalg.norm(img_f - key[None, None, :], axis=2)

print("RGB distance statistics")
for p in [0, 1, 5, 25, 50, 75, 95, 99, 100]:
    print(f"p{p:>3}: {np.percentile(rgb_dist, p):.2f}")

plt.figure(figsize=(14, 4))
plt.hist(rgb_dist.ravel(), bins=200)
plt.title("Histogram: RGB distance to chroma key")
plt.xlabel("RGB distance")
plt.ylabel("pixels");

## Baseline method: RGB distance threshold

This is the simplest method: pixels close to magenta are background.  
It often fails near anti-aliased borders because border pixels are neither pure magenta nor pure object color.

In [ ]:
# Tune this first.
RGB_THRESHOLD = 45

bg_mask_rgb = rgb_dist < RGB_THRESHOLD
fg_mask_rgb = ~bg_mask_rgb

plt.figure(figsize=(14, 7))
plt.imshow(bg_mask_rgb, cmap="gray")
plt.title(f"Background mask, RGB distance < {RGB_THRESHOLD}")
plt.axis("off");

rgba_rgb = np.dstack([img, (fg_mask_rgb.astype(np.uint8) * 255)])
plt.figure(figsize=(14, 7))
plt.imshow(rgba_rgb)
plt.title("RGBA preview from RGB threshold")
plt.axis("off");

## HSV diagnostic

HSV can be useful because the key color is strongly saturated and has a distinctive hue.  
However, hue becomes unstable near very dark or very bright pixels, so use it together with saturation/value constraints.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
H, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]

# OpenCV hue is 0..179. Magenta #FF00FF is around hue 150.
key_hsv = cv2.cvtColor(np.uint8([[KEY_RGB]]), cv2.COLOR_RGB2HSV)[0, 0]
print("KEY HSV:", key_hsv)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, channel, title in zip(axes, [H, S, V], ["Hue", "Saturation", "Value"]):
    im = ax.imshow(channel, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## HSV keying

Useful when the background is almost magenta but not exactly `#FF00FF`.

For magenta:
- hue should be near 150 in OpenCV HSV,
- saturation should be high,
- value should be high.

This method may still produce halos if the generated image creates blended edges.

In [ ]:
KEY_H = int(key_hsv[0])

# Tune these.
HUE_TOL = 12
MIN_S = 80
MIN_V = 80

# Circular hue distance in OpenCV's 0..179 hue space.
hue_dist = np.minimum(np.abs(H.astype(np.int16) - KEY_H), 180 - np.abs(H.astype(np.int16) - KEY_H))

bg_mask_hsv = (hue_dist <= HUE_TOL) & (S >= MIN_S) & (V >= MIN_V)
fg_mask_hsv = ~bg_mask_hsv

plt.figure(figsize=(14, 7))
plt.imshow(bg_mask_hsv, cmap="gray")
plt.title(f"Background mask HSV: hue±{HUE_TOL}, S>={MIN_S}, V>={MIN_V}")
plt.axis("off");

rgba_hsv = np.dstack([img, (fg_mask_hsv.astype(np.uint8) * 255)])
plt.figure(figsize=(14, 7))
plt.imshow(rgba_hsv)
plt.title("RGBA preview from HSV mask")
plt.axis("off");

## Combine RGB and HSV masks

A conservative background mask can reduce false removals.  
For chroma-key extraction, false positives are often worse than false negatives: a remaining pink fringe is easier to post-process than missing object detail.

In [ ]:
# Candidate strategy:
# background if it is close in RGB OR it strongly matches magenta in HSV.
bg_mask = bg_mask_rgb | bg_mask_hsv

plt.figure(figsize=(14, 7))
plt.imshow(bg_mask, cmap="gray")
plt.title("Combined background mask")
plt.axis("off");

## Keep only background connected to the image border

The combined mask above flags every chroma-like pixel — including ones that
happen to share that colour *inside* the silhouette (a magenta dress on a
magenta background, foliage that shares hue with a green key, etc.). True
background must be reachable from the image edge, so we keep only the
connected components of `bg_mask` that touch a border row or column. The
rest stays foreground.


In [ ]:
def background_connected_to_border(bg_mask: np.ndarray) -> np.ndarray:
    """Connected components of `bg_mask` that touch the image border."""
    num, labels = cv2.connectedComponents(bg_mask.astype(np.uint8), connectivity=4)
    if num <= 1:
        return np.zeros_like(bg_mask, dtype=bool)
    # `labels` is 0 outside `bg_mask`, so the values on each border row /
    # column tell us which components reach the edge.
    border_labels = set()
    border_labels.update(np.unique(labels[0]).tolist())
    border_labels.update(np.unique(labels[-1]).tolist())
    border_labels.update(np.unique(labels[:, 0]).tolist())
    border_labels.update(np.unique(labels[:, -1]).tolist())
    border_labels.discard(0)
    keep = np.isin(labels, list(border_labels))
    return keep


bg_mask_border = background_connected_to_border(bg_mask)
interior_kept = bg_mask & ~bg_mask_border

print(f"Combined background mask:        {int(bg_mask.sum()):>10} px")
print(f"Reachable from border (kept):    {int(bg_mask_border.sum()):>10} px")
print(f"Interior chroma preserved as FG: {int(interior_kept.sum()):>10} px")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(bg_mask, cmap="gray")
axes[0].set_title("Combined mask (every chroma-like pixel)")
axes[0].axis("off")
axes[1].imshow(bg_mask_border, cmap="gray")
axes[1].set_title("Border-connected background (what we'll remove)")
axes[1].axis("off")
axes[2].imshow(interior_kept, cmap="hot")
axes[2].set_title("Interior chroma preserved as foreground")
axes[2].axis("off")
plt.tight_layout()
plt.show()

# Replace `bg_mask` so downstream cells use the border-restricted version.
bg_mask = bg_mask_border


## Mask cleanup

Common cleanup operations:
- **opening** removes small isolated background detections inside the object;
- **closing** fills small holes in the background mask;
- **dilation** expands the background mask, useful to remove a chroma halo;
- **erosion** preserves the object edge but may leave halo.

For foliage and irregular edges, be careful: too much morphology destroys detail.

In [ ]:
# Tune these.
OPEN_K = 0       # 0 disables
CLOSE_K = 0      # 0 disables
DILATE_K = 1     # useful for removing magenta fringe
ERODE_K = 0

def morph_mask(mask, open_k=0, close_k=0, dilate_k=0, erode_k=0):
    m = mask.astype(np.uint8) * 255
    if open_k > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    if close_k > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    if dilate_k > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_k, dilate_k))
        m = cv2.dilate(m, k)
    if erode_k > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_k, erode_k))
        m = cv2.erode(m, k)
    return m > 0

bg_clean = morph_mask(bg_mask, OPEN_K, CLOSE_K, DILATE_K, ERODE_K)
fg_clean = ~bg_clean

plt.figure(figsize=(14, 7))
plt.imshow(bg_clean, cmap="gray")
plt.title("Cleaned background mask")
plt.axis("off");

## Soft alpha from distance transform

A binary mask creates jagged edges.  
This step creates a gradual alpha around the boundary.

Interpretation:
- `FEATHER_PX`: width of the transition zone.
- Too low: jagged edge.
- Too high: ghostly semi-transparent borders.

In [ ]:
FEATHER_PX = 2.0

# Distance to nearest background pixel, computed inside foreground.
fg_u8 = fg_clean.astype(np.uint8)
dist_inside_fg = cv2.distanceTransform(fg_u8, cv2.DIST_L2, 3)

alpha = np.clip(dist_inside_fg / FEATHER_PX, 0, 1)
alpha[bg_clean] = 0.0

rgba_soft = np.dstack([img, (alpha * 255).astype(np.uint8)])

plt.figure(figsize=(14, 7))
plt.imshow(alpha, cmap="gray")
plt.title(f"Soft alpha, feather={FEATHER_PX}px")
plt.axis("off");

plt.figure(figsize=(14, 7))
plt.imshow(rgba_soft)
plt.title("RGBA preview with soft alpha")
plt.axis("off");

## Despill / chroma fringe reduction

Antialiased edge pixels mix the foreground colour with the chroma colour, so
the silhouette band carries leftover chroma even after the alpha is right.

The despill below is **chroma-direction-aware**: it inspects `KEY_RGB`,
identifies its dominant channels (the ones strongly above the other(s)) and
suppresses the "excess" of those channels over the suppressed one(s) at
edge pixels. Result:

* magenta key `(255, 0, 255)` → pulls R and B toward G (classic pink fringe);
* green key `(0, 255, 0)` → pulls G toward `max(R, B)` (classic green fringe);
* cyan / blue / yellow / red keys handled the same way;
* near-grey keys (no chroma direction, e.g. beige `(240, 224, 211)`) are
  left alone — there's nothing to pull away from. The residual QA cell
  below catches whatever halo remains and you can address it with mask
  dilation or `RESIDUAL_FIX`.

`DESPILL_STRENGTH = 0` disables it entirely.


In [ ]:
DESPILL_STRENGTH = 0.7  # 0 disables, 1 strong


def despill_for_key(
    rgb_linear: np.ndarray,
    alpha: np.ndarray,
    key_rgb: np.ndarray,
    strength: float,
) -> np.ndarray:
    """Chroma-direction-aware despill at the silhouette edge band.

    For a saturated key the channels with value >= 50% of the key's max are
    the "dominant" ones; the rest are "suppressed". An edge pixel whose
    dominant-channel minimum exceeds its suppressed-channel maximum carries
    chroma spill in the dominant direction, which we shrink. A
    near-grey/near-black key has no chroma direction and we skip — rely on
    `RESIDUAL_FIX` below for those.
    """
    out = rgb_linear.copy()
    if strength <= 0.0:
        return out
    edge = (alpha > 0.0) & (alpha < 1.0)
    if not edge.any():
        return out
    key = key_rgb.astype(np.float32) / 255.0
    max_k = float(key.max())
    if max_k < 1e-3:
        return out
    dom_mask = key >= 0.5 * max_k
    dom_chans = [c for c in range(3) if dom_mask[c]]
    sup_chans = [c for c in range(3) if not dom_mask[c]]
    if not dom_chans or not sup_chans:
        return out  # no chroma direction (near-grey/white/black key)
    dom_min = rgb_linear[..., dom_chans].min(axis=-1)
    sup_max = rgb_linear[..., sup_chans].max(axis=-1)
    excess = np.maximum(0.0, dom_min - sup_max)
    correction = strength * excess * edge.astype(np.float32)
    for c in dom_chans:
        out[..., c] = np.clip(out[..., c] - correction, 0.0, 1.0)
    for c in sup_chans:
        # Gentle compensation keeps perceived luminance roughly constant.
        out[..., c] = np.clip(out[..., c] + 0.25 * correction, 0.0, 1.0)
    return out


rgb = img.astype(np.float32) / 255.0
rgb_despill = despill_for_key(rgb, alpha, KEY_RGB, DESPILL_STRENGTH)

rgba_despill = np.dstack(
    [(rgb_despill * 255).astype(np.uint8), (alpha * 255).astype(np.uint8)]
)

plt.figure(figsize=(14, 7))
plt.imshow(rgba_despill)
plt.title(
    f"RGBA preview with despill (key={tuple(int(x) for x in KEY_RGB)}, "
    f"strength={DESPILL_STRENGTH})"
)
plt.axis("off")
plt.show()


## Final QA: chroma residuals on the silhouette edge

Any pixel in the transition band (`0 < alpha < 1`) that still sits close to
the chroma colour is a residual halo. We measure:

* the size of the edge band,
* distance-to-key (mean and p95) within the band, before and after despill,
* a heatmap of residual pixels under `RESIDUAL_DIST_TOL`.

Toggle `RESIDUAL_FIX` to hard-zero alpha on detected residuals at the cost
of slightly thinner edges. Run with `False` first to see what the rest of
the pipeline left behind.


In [ ]:
RESIDUAL_DIST_TOL = 60.0  # RGB distance under which an edge pixel counts as residual
RESIDUAL_FIX = False      # True: zero alpha where a residual is detected

edge_band = (alpha > 0.0) & (alpha < 1.0)
edge_count = int(edge_band.sum())

key_f = KEY_RGB.astype(np.float32)
dist_pre = np.linalg.norm(img.astype(np.float32) - key_f[None, None, :], axis=2)
dist_post = np.linalg.norm(
    (rgb_despill * 255.0) - key_f[None, None, :], axis=2
)

if edge_count > 0:
    pre = dist_pre[edge_band]
    post = dist_post[edge_band]
    print(
        f"Edge band: {edge_count} px "
        f"({100*edge_count/edge_band.size:.2f}% of image)"
    )
    print(
        f"  distance-to-key, pre-despill : mean={pre.mean():.1f}, "
        f"p95={np.percentile(pre, 95):.1f}"
    )
    print(
        f"  distance-to-key, post-despill: mean={post.mean():.1f}, "
        f"p95={np.percentile(post, 95):.1f}"
    )
    residual = edge_band & (dist_post < RESIDUAL_DIST_TOL)
    print(
        f"Residual (distance < {RESIDUAL_DIST_TOL}): {int(residual.sum())} px "
        f"({100*residual.sum()/max(edge_count,1):.2f}% of edge band)"
    )
else:
    print("No edge band — alpha is fully binary; consider raising FEATHER_PX.")
    residual = np.zeros_like(edge_band)

# Heatmap: brighter = closer to the chroma key.
res_view = np.where(residual, np.clip(255.0 - dist_post, 0, 255), 0).astype(np.uint8)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(edge_band, cmap="gray")
axes[0].set_title("Edge band (0 < alpha < 1)")
axes[0].axis("off")
axes[1].imshow(res_view, cmap="hot")
axes[1].set_title(f"Residuals (distance < {RESIDUAL_DIST_TOL})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

alpha_final = alpha.copy()
if RESIDUAL_FIX:
    alpha_final[residual] = 0.0

rgba_final = np.dstack(
    [
        (rgb_despill * 255).astype(np.uint8),
        (alpha_final * 255).astype(np.uint8),
    ]
)


## Preview over test backgrounds

A good extraction should look acceptable over several backgrounds.  
Use this to catch halos that are invisible over a similar color.

In [ ]:
def composite_over_rgba(rgba, bg_rgb):
    a = rgba[..., 3:4].astype(np.float32) / 255.0
    fg = rgba[..., :3].astype(np.float32)
    bg = np.zeros_like(fg) + np.array(bg_rgb, dtype=np.float32)
    out = fg * a + bg * (1 - a)
    return out.astype(np.uint8)

test_backgrounds = {
    "dark gray": (40, 40, 40),
    "light gray": (210, 210, 210),
    "sky blue": (110, 170, 220),
    "earth brown": (130, 95, 60),
}

for name, bg in test_backgrounds.items():
    preview = composite_over_rgba(rgba_final, bg)
    plt.figure(figsize=(14, 7))
    plt.imshow(preview)
    plt.title(f"Preview over {name}")
    plt.axis("off")
    plt.show()

## Crop the canvas to the silhouette

After extraction the foreground typically occupies only part of the original
canvas, leaving large transparent margins. Crop to the bounding box of
opaque-enough pixels (plus optional padding) so the output PNG is no
larger than it needs to be — the game's asset pipeline expects tight art.


In [ ]:
def crop_to_content(
    rgba: np.ndarray, alpha_threshold: int = 5, padding: int = 0
) -> np.ndarray:
    """Crop to the bounding box of pixels with alpha > threshold (+ padding)."""
    a = rgba[..., 3]
    mask = a > alpha_threshold
    if not np.any(mask):
        return rgba
    ys, xs = np.where(mask)
    y0 = max(int(ys.min()) - padding, 0)
    y1 = min(int(ys.max()) + padding + 1, rgba.shape[0])
    x0 = max(int(xs.min()) - padding, 0)
    x1 = min(int(xs.max()) + padding + 1, rgba.shape[1])
    return rgba[y0:y1, x0:x1]


CROP_ALPHA_THRESHOLD = 5  # pixels less opaque than this are treated as background for the bbox
CROP_PADDING = 0          # add N transparent pixels around the bbox (useful for tight art)

before_h, before_w = rgba_final.shape[:2]
rgba_cropped = crop_to_content(rgba_final, CROP_ALPHA_THRESHOLD, CROP_PADDING)
after_h, after_w = rgba_cropped.shape[:2]
ratio = 100.0 * (after_h * after_w) / max(before_h * before_w, 1)
print(
    f"Canvas: {before_w}x{before_h}  ->  {after_w}x{after_h}  ({ratio:.1f}% of original)"
)

plt.figure(figsize=(14, 7))
plt.imshow(rgba_cropped)
plt.title(f"Cropped RGBA ({after_w}x{after_h})")
plt.axis("off")
plt.show()


## Save output

In [ ]:
OUT_PATH = IMAGE_PATH.with_name(IMAGE_PATH.stem + "_extracted.png")
cv2.imwrite(str(OUT_PATH), cv2.cvtColor(rgba_cropped, cv2.COLOR_RGBA2BGRA))
print("Saved:", OUT_PATH.resolve())

## What to look for when it fails

Typical symptoms and likely causes:

| Symptom | Likely cause | Parameter to inspect |
|---|---|---|
| Chroma color seems off | borders contained content / sampled too small | set `KEY_RGB_OVERRIDE`, raise `PATCH` |
| Interior chroma-coloured regions punched out | flood-fill found a path from a hole through to the border (e.g. background leaks into the silhouette) | shrink `RGB_THRESHOLD` / `HUE_TOL`, increase `CLOSE_K` |
| Pink halo remains | background mask too conservative | increase `RGB_THRESHOLD`, `HUE_TOL`, or `DILATE_K` |
| Object edge is eaten | background mask too aggressive | decrease `RGB_THRESHOLD`, `HUE_TOL`, or `DILATE_K` |
| Holes inside object | key color too close to object colors, or threshold too loose | lower thresholds, use conservative combined mask |
| Foliage looks damaged | key color conflicts with vegetation, or morphology too strong | avoid green key; lower morphology |
| Jagged edge | binary alpha or too little feathering | increase `FEATHER_PX` |
| Ghostly transparent edge | feathering too wide | decrease `FEATHER_PX` |
| Magenta contamination on border | anti-aliasing/despill issue | increase `DESPILL_STRENGTH`, slightly dilate background mask |
| Residual halo reported above | despill didn't clear the band, halo masquerading as soft alpha | enable `RESIDUAL_FIX`, raise `RESIDUAL_DIST_TOL`, or dilate background mask |
| Output PNG much bigger than the silhouette | wide transparent margins | already auto-cropped; lower `CROP_ALPHA_THRESHOLD` if the bbox loses very-faint edges |
